# Smart Control Course Project
## Optimization-Based Controller Tuning: PID, Neural Network, Fuzzy Logic & Genetic Programming

**Optimizers (from scratch):** Genetic Algorithm (GA), Differential Evolution (DE), Particle Swarm Optimization (PSO), Simulated Annealing (SA), Genetic Programming (GP)  
**Controllers:** PID, Neural Network, Fuzzy Logic, GP Controller  
**Plants:** 2nd-order, 3rd-order, 4th-order  
**Performance Metrics:** Rise Time, Settling Time, Overshoot, Steady-State Error, IAE  

---
**Libraries allowed:** `control`, `numpy`, `matplotlib` (all algorithms implemented from scratch)

## 1. Setup & Imports

In [ ]:
# Install required libraries (for Google Colab)
# !pip install control numpy matplotlib

import math
import random
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import control as ct

# Global simulation settings
DT        = 0.05       # time step (s) — larger step for speed
T_END     = 20.0       # simulation duration (s)
REFERENCE = 1.0        # unit step setpoint
U_LIMIT   = 50.0       # control signal saturation
Y_LIMIT   = 1e3        # output divergence guard
SETTLING_TOL = 0.02    # ±2% settling band

print("Imports successful ✓")
print(f"Python control version: {ct.__version__}")

: 

---
## 2. Plant Models

Three continuous-time transfer-function plants are used:

| Plant | Transfer Function | Poles |
|-------|------------------|-------|
| 2nd-order | $G(s) = \dfrac{1}{s^2 + 1.2s + 1}$ | $s = -0.6 \pm j0.8$ (underdamped) |
| 3rd-order | $G(s) = \dfrac{10}{s^3 + 6s^2 + 11s + 6}$ | $s = -1, -2, -3$ (real) |
| 4th-order | $G(s) = \dfrac{1}{s^4 + 2.6s^3 + 3.44s^2 + 2.4s + 0.64}$ | mixed (stable) |

In [ ]:
def get_plants():
    """Return dict of plant transfer functions."""
    return {
        'second_order': ct.tf([1],       [1, 1.2, 1]),
        'third_order':  ct.tf([10],      [1, 6, 11, 6]),
        'fourth_order': ct.tf([1],       [1, 2.6, 3.44, 2.4, 0.64]),
    }

PLANTS = get_plants()

# Show pole locations
for name, plant in PLANTS.items():
    poles = ct.poles(plant)
    print(f"{name}: poles = {poles}")

---
## 3. Closed-Loop Simulation

Unity negative feedback using Zero-Order-Hold (ZOH) discretization:

$$x[k+1] = A_d x[k] + B_d u[k], \quad y[k] = C_d x[k]$$
$$e[k] = r - y[k], \quad u[k] = \text{controller}(e[k], \Delta t)$$

In [ ]:
# Cache for discretized plants
_disc_cache = {}

def _discretize(plant, dt):
    key = (id(plant), dt)
    if key not in _disc_cache:
        sys_ss = ct.ss(plant)
        sysd   = ct.c2d(sys_ss, dt, method='zoh')
        _disc_cache[key] = (
            np.asarray(sysd.A, float),
            np.asarray(sysd.B, float).ravel(),
            np.asarray(sysd.C, float).ravel(),
        )
    return _disc_cache[key]

def simulate(plant, controller, dt=DT, t_end=T_END, ref=REFERENCE):
    """Simulate closed-loop unit-step response. Returns (time, output, control)."""
    Ad, Bd, Cd = _discretize(plant, dt)
    n_steps = int(round(t_end / dt))
    time_arr = np.arange(n_steps) * dt

    controller.reset()
    x = np.zeros(Ad.shape[0])
    y_arr = np.empty(n_steps)
    u_arr = np.empty(n_steps)

    for k in range(n_steps):
        y = float(Cd @ x)
        if not math.isfinite(y) or abs(y) > Y_LIMIT:
            y_arr[k:] = np.nan
            u_arr[k:] = np.nan
            break
        e = ref - y
        u = float(controller.compute(e, dt))
        if not math.isfinite(u):
            y_arr[k:] = np.nan; u_arr[k:] = np.nan; break
        u = max(-U_LIMIT, min(U_LIMIT, u))
        y_arr[k] = y
        u_arr[k] = u
        x = Ad @ x + Bd * u
    return time_arr, y_arr, u_arr

print("Simulation function ready ✓")

---
## 4. Performance Metrics

Five standard control-engineering metrics:

| Metric | Formula / Definition |
|--------|---------------------|
| Rise Time $T_r$ | Time to go from 10% to 90% of reference |
| Settling Time $T_s$ | First time output stays within $\pm 2\%$ band forever |
| Overshoot | $OS = \frac{y_{\max} - r}{r} \times 100\%$ |
| Steady-State Error | $e_{ss} = |r - \bar{y}_{\text{final}}|$ |
| IAE | $\int_0^T |e(t)| \, dt$ |

In [ ]:
_trapz = np.trapezoid if hasattr(np, 'trapezoid') else np.trapz

def compute_metrics(t, y, ref=REFERENCE, tol=SETTLING_TOL):
    """Return dict of performance metrics."""
    valid = np.isfinite(y)
    if not np.any(valid):
        return dict(rise_time=None, settling_time=None, overshoot=float('inf'),
                    sse=float('inf'), iae=float('inf'), stable=False)

    # IAE
    err = ref - y
    err_safe = np.where(valid, np.abs(err), 0.0)
    iae = float(_trapz(err_safe, t))

    # Overshoot
    y_safe = np.where(valid, y, -np.inf)
    y_max  = float(np.max(y_safe))
    overshoot = max(0.0, (y_max - ref) / (abs(ref) + 1e-12) * 100.0)

    # Steady-state error (average over last 10% of valid signal)
    n_valid = int(np.sum(valid))
    tail = max(1, n_valid // 10)
    y_valid = y[valid]
    sse = float(abs(ref - np.mean(y_valid[-tail:])))

    # Rise time (10% → 90%)
    lo, hi = 0.1 * ref, 0.9 * ref
    rise_time = None
    t_lo = t_hi = None
    for i in range(len(t) - 1):
        if valid[i] and valid[i+1]:
            if t_lo is None and y[i] <= lo <= y[i+1]:
                frac = (lo - y[i]) / (y[i+1] - y[i] + 1e-15)
                t_lo = t[i] + frac * (t[i+1] - t[i])
            if t_hi is None and y[i] <= hi <= y[i+1]:
                frac = (hi - y[i]) / (y[i+1] - y[i] + 1e-15)
                t_hi = t[i] + frac * (t[i+1] - t[i])
    if t_lo is not None and t_hi is not None:
        rise_time = max(0.0, t_hi - t_lo)

    # Settling time
    band = tol * abs(ref)
    settling_time = None
    for i in range(len(t) - 1, -1, -1):
        if valid[i] and abs(y[i] - ref) > band:
            if i + 1 < len(t) and valid[i+1]:
                settling_time = float(t[i+1])
            break
    else:
        settling_time = float(t[0])  # always within band

    return dict(rise_time=rise_time, settling_time=settling_time,
                overshoot=overshoot, sse=sse, iae=iae, stable=True)

def scalar_cost(metrics, penalty=1e4):
    """Weighted scalar objective (minimized by optimizers)."""
    if not metrics['stable'] or not math.isfinite(metrics['iae']):
        return float(penalty)
    ts = metrics['settling_time'] if metrics['settling_time'] else T_END * 1.5
    tr = metrics['rise_time']     if metrics['rise_time']     else T_END * 1.5
    return (1.0 * metrics['iae']
          + 0.5 * metrics['overshoot']
          + 0.3 * ts
          + 0.2 * tr
          + 0.3 * metrics['sse'])

print("Metrics functions ready ✓")

---
## 5. Controllers
### 5.1 PID Controller

$$u(t) = K_p \, e(t) + K_i \int e(t)\,dt + K_d \frac{de}{dt}$$

Discrete implementation:
- Integral: $I[k] = I[k-1] + e[k] \cdot \Delta t$ (anti-windup clamp $\pm 50$)
- Derivative: $D[k] = (e[k] - e[k-1]) / \Delta t$ (0 on first step)

Optimized parameters: $[K_p, K_i, K_d]$

In [ ]:
class PIDController:
    """Discrete PID with anti-windup and output saturation."""
    name = 'PID'

    def __init__(self):
        self.Kp = 1.0; self.Ki = 0.0; self.Kd = 0.0
        self._int_limit = 50.0
        self.reset()

    def reset(self):
        """Clear integrator and derivative state."""
        self._integral  = 0.0
        self._prev_err  = 0.0
        self._has_prev  = False

    def set_parameters(self, params):
        self.Kp, self.Ki, self.Kd = float(params[0]), float(params[1]), float(params[2])

    def compute(self, error, dt):
        """u = Kp*e + Ki*∫e*dt + Kd*(de/dt)"""
        error = float(error)

        # Integral with anti-windup
        self._integral += error * dt
        self._integral = max(-self._int_limit, min(self._int_limit, self._integral))

        # Derivative (backward difference)
        if self._has_prev:
            derivative = (error - self._prev_err) / dt
        else:
            derivative = 0.0
        self._prev_err = error
        self._has_prev = True

        u = self.Kp * error + self.Ki * self._integral + self.Kd * derivative
        return max(-U_LIMIT, min(U_LIMIT, u))

    def parameter_bounds(self):
        return [(0.0, 50.0), (0.0, 20.0), (0.0, 10.0)]  # [Kp, Ki, Kd]

print("PID Controller ready ✓")

### 5.2 Neural Network Controller

Feed-forward network: **2 → 5 → 1**
- Input: $[e(t), e(t-1)]$ — current and previous error
- Hidden layer: 5 neurons, **tanh** activation
- Output: 1 neuron, **linear** activation
- Total parameters: $2\times5 + 5 + 5\times1 + 1 = 21$

Forward pass:
$$h = \tanh(W_1 x + b_1), \quad u = W_2 h + b_2$$

In [ ]:
class NeuralNetworkController:
    """2→5→1 feed-forward neural network controller."""
    name = 'Neural Network'

    def __init__(self):
        self.n_in, self.n_hid, self.n_out = 2, 5, 1
        # Total params: (2*5 + 5) + (5*1 + 1) = 15 + 6 = 21
        self.n_params = self.n_in*self.n_hid + self.n_hid + self.n_hid*self.n_out + self.n_out
        self._init_weights()
        self.prev_error = 0.0

    def _init_weights(self):
        rng = np.random.default_rng(42)
        self.W1 = rng.standard_normal((self.n_hid, self.n_in)) * 0.1
        self.b1 = rng.standard_normal(self.n_hid) * 0.1
        self.W2 = rng.standard_normal((self.n_out, self.n_hid)) * 0.1
        self.b2 = rng.standard_normal(self.n_out) * 0.1

    def reset(self):
        """Only reset dynamic state (prev_error), not weights."""
        self.prev_error = 0.0

    def set_parameters(self, params):
        """Unpack flat vector [W1, b1, W2, b2]."""
        p = np.array(params, dtype=float)
        idx = 0
        n = self.n_in * self.n_hid
        self.W1 = p[idx:idx+n].reshape(self.n_hid, self.n_in); idx += n
        self.b1 = p[idx:idx+self.n_hid]; idx += self.n_hid
        n = self.n_hid * self.n_out
        self.W2 = p[idx:idx+n].reshape(self.n_out, self.n_hid); idx += n
        self.b2 = p[idx:idx+self.n_out]

    def compute(self, error, dt):
        """Forward pass: input = [e(t), e(t-1)]."""
        x = np.array([float(error), self.prev_error])
        h = np.tanh(self.W1 @ x + self.b1)      # hidden: (5,)
        u = float((self.W2 @ h + self.b2)[0])    # output: scalar
        self.prev_error = float(error)
        return max(-U_LIMIT, min(U_LIMIT, u))

    def parameter_bounds(self):
        return [(-5.0, 5.0)] * self.n_params  # all weights in [-5, 5]

print(f"Neural Network Controller ready ✓  (n_params = {NeuralNetworkController().n_params})")

### 5.3 Fuzzy Logic Controller (Mamdani)

**Two inputs:** normalized error $e_n$ and normalized error rate $\dot{e}_n$  
**Five triangular membership functions:** NB, NS, ZE, PS, PB (centers at $-1, -0.5, 0, 0.5, 1$)  
**25-rule table** (output index 0–4 → NB to PB):  
**Defuzzification:** centroid method

Optimized parameters: $[K_e, K_{de}, K_u]$ — input/output scaling gains

In [ ]:
class FuzzyController:
    """Two-input Mamdani fuzzy controller with 5×5 rule base."""
    name = 'Fuzzy'

    _CENTERS = np.array([-1.0, -0.5, 0.0, 0.5, 1.0])
    # Rule table [row=error, col=error_rate] → output set index (0=NB ... 4=PB)
    _RULES = np.array([
        [0, 0, 0, 1, 2],
        [0, 0, 1, 2, 3],
        [0, 1, 2, 3, 4],
        [1, 2, 3, 4, 4],
        [2, 3, 4, 4, 4],
    ], dtype=int)

    def __init__(self):
        self.K_e  = 2.0    # error scaling
        self.K_de = 0.1    # error-rate scaling
        self.K_u  = 10.0   # output scaling
        # Pre-compute output membership matrix over 201-point grid
        self._grid = np.linspace(-1.0, 1.0, 201)
        self._out_mem = self._memberships(self._grid)
        self.reset()

    @classmethod
    def _memberships(cls, values):
        """Triangular MFs: width 0.5 between adjacent centers."""
        v = np.asarray(values).reshape(-1, 1)
        c = cls._CENTERS.reshape(1, -1)
        return np.clip(1.0 - np.abs(v - c) / 0.5, 0.0, 1.0)

    def reset(self):
        self._prev_err = 0.0
        self._has_prev = False

    def set_parameters(self, params):
        self.K_e, self.K_de, self.K_u = float(params[0]), float(params[1]), float(params[2])

    def compute(self, error, dt):
        """Mamdani inference with centroid defuzzification."""
        error = float(error)
        # Error rate
        if self._has_prev:
            de = (error - self._prev_err) / dt
        else:
            de = 0.0
        self._prev_err = error; self._has_prev = True

        # Normalize inputs
        e_n  = float(np.clip(self.K_e  * error, -1.0, 1.0))
        de_n = float(np.clip(self.K_de * de,    -1.0, 1.0))

        # Fuzzify
        mu_e  = self._memberships([e_n])[0]
        mu_de = self._memberships([de_n])[0]

        # Mamdani: min-implication, max-aggregation
        agg = np.zeros_like(self._grid)
        for i in range(5):
            for j in range(5):
                strength = min(mu_e[i], mu_de[j])
                if strength > 0:
                    out_set = self._out_mem[:, self._RULES[i, j]]
                    agg = np.maximum(agg, np.minimum(strength, out_set))

        # Centroid defuzzification
        area = float(_trapz(agg, self._grid))
        if area <= 1e-12:
            return 0.0
        centroid = float(_trapz(self._grid * agg, self._grid)) / area
        return float(self.K_u * centroid)

    def parameter_bounds(self):
        return [(0.1, 20.0), (0.001, 5.0), (0.1, 50.0)]  # [K_e, K_de, K_u]

print("Fuzzy Controller ready ✓")

### 5.4 Genetic Programming Controller

An expression tree over three signals is evolved and evaluated as the control law:

| Signal | Description |
|--------|-------------|
| $e$    | Tracking error $r - y$ |
| $de$   | Error rate $(e[k]-e[k-1])/\Delta t$ |
| $ie$   | Error integral $\sum e \cdot \Delta t$ (clamped ±50) |

**Function set:** `add`, `sub`, `mul`, `div` (protected), `tanh`  
**Example tree (PID-like law):** `add(add(mul(Kp, e), mul(Ki, ie)), mul(Kd, de))`

In [ ]:
# --- GP Tree utilities ---
ARITY = {'add': 2, 'sub': 2, 'mul': 2, 'div': 2, 'tanh': 1}
VARS  = ('e', 'de', 'ie')
FUNCS = tuple(ARITY.keys())
GP_MAX_DEPTH = 3
GP_CONST_RANGE = (-10.0, 10.0)

def _pdiv(a, b): return a / b if abs(b) > 1e-6 else 1.0

def eval_tree(tree, e, de, ie):
    """Recursively evaluate a GP tree."""
    if isinstance(tree, tuple):
        name, *children = tree
        args = [eval_tree(c, e, de, ie) for c in children]
        if name == 'add':  return args[0] + args[1]
        if name == 'sub':  return args[0] - args[1]
        if name == 'mul':  return args[0] * args[1]
        if name == 'div':  return _pdiv(args[0], args[1])
        if name == 'tanh': return math.tanh(args[0])
    if isinstance(tree, str):
        return {'e': e, 'de': de, 'ie': ie}[tree]
    return float(tree)

def tree_to_str(tree):
    """Human-readable tree string."""
    if isinstance(tree, tuple):
        name, *children = tree
        return f"{name}({', '.join(tree_to_str(c) for c in children)})"
    if isinstance(tree, float):
        return f"{tree:.3g}"
    return str(tree)


class GPController:
    """Evaluates a GP expression tree as a control law."""
    name = 'GP Controller'

    def __init__(self, tree=None):
        self._tree = tree
        self.reset()

    def reset(self):
        self._prev_e = 0.0
        self._ie     = 0.0
        self._has_prev = False

    def set_tree(self, tree):
        self._tree = tree

    def set_parameters(self, params):
        """Decode a numeric vector into a tree (heap layout, depth≤3)."""
        self._tree = _decode_gp_vector(np.asarray(params, float))

    def compute(self, error, dt):
        if self._tree is None:
            return 0.0
        error = float(error)
        # Derivative
        de = (error - self._prev_e) / dt if self._has_prev else 0.0
        # Integral (anti-windup)
        self._ie = max(-50.0, min(50.0, self._ie + error * dt))
        self._prev_e = error; self._has_prev = True
        try:
            raw = eval_tree(self._tree, error, de, self._ie)
        except Exception:
            return 0.0
        if not math.isfinite(raw): return 0.0
        return max(-U_LIMIT, min(U_LIMIT, raw))

    def parameter_bounds(self):
        """Bounds for numeric GP vector encoding."""
        return _gp_parameter_bounds(GP_MAX_DEPTH)


# --- Numeric vector encoding for GP trees (heap layout) ---
# Slot i has children 2i+1 and 2i+2. Each slot has 2 genes: [sym_idx+0.5, constant].
# Interior slots (depth < MAX_DEPTH): symbol from FUNCS + VARS + ['const']
# Leaf slots (depth == MAX_DEPTH): symbol from VARS + ['const']

def _slot_depth(slot):
    d = 0; s = slot
    while s > 0: s = (s - 1) // 2; d += 1
    return d

def _gp_parameter_bounds(max_depth):
    n_slots = 2**(max_depth + 1) - 1
    n_funcs = len(FUNCS); n_vars = len(VARS)
    bounds = []
    for s in range(n_slots):
        is_leaf = _slot_depth(s) == max_depth
        n_sym = n_vars + 1 if is_leaf else n_funcs + n_vars + 1
        bounds.append((0.5, n_sym - 0.5))  # symbol index
        bounds.append(GP_CONST_RANGE)       # constant value
    return bounds

def _decode_gp_vector(vec, max_depth=GP_MAX_DEPTH):
    n_funcs = len(FUNCS); n_vars = len(VARS)
    all_sym  = list(FUNCS) + list(VARS) + ['const']
    leaf_sym = list(VARS)  + ['const']
    lo, hi = GP_CONST_RANGE

    def build(slot):
        is_leaf = _slot_depth(slot) == max_depth
        g_sym = vec[2 * slot]
        g_const = max(lo, min(hi, float(vec[2 * slot + 1])))
        syms = leaf_sym if is_leaf else all_sym
        idx  = max(0, min(len(syms) - 1, int(g_sym)))
        sym  = syms[idx]
        if sym == 'const': return g_const
        if sym in ARITY:
            children = tuple(build(2*slot+1+k) for k in range(ARITY[sym]))
            return (sym,) + children
        return sym  # variable

    return build(0)


print(f"GP Controller ready ✓  (parameter vector length = {len(GPController().parameter_bounds())})")

---
## 6. Optimization Algorithms (from scratch)

### 6.1 Objective Function Factory

In [ ]:
PENALTY = 1e4

def make_objective(controller, plant):
    """Return f(params) → scalar cost (to be minimized)."""
    def objective(params):
        try:
            p = np.asarray(params, dtype=float)
            if not np.all(np.isfinite(p)): return PENALTY
            controller.set_parameters(p)
            t, y, _ = simulate(plant, controller)
            m = compute_metrics(t, y)
            return scalar_cost(m, PENALTY)
        except Exception:
            return PENALTY
    return objective

print("Objective factory ready ✓")

### 6.2 Genetic Algorithm (GA)

Following the **course lecture on Real-Valued GA**:

| Step | Description |
|------|-------------|
| 1. Initialize | Uniform random population in bounds |
| 2. Evaluate   | Calculate objective for each individual |
| 3. Select     | **Tournament selection** (k=2) |
| 4. Crossover  | **Arithmetic crossover**: $c = \alpha p_1 + (1-\alpha)p_2$, $p_c = 0.9$ |
| 5. Mutate     | **Additive (Gaussian) mutation**: $x' = x + \epsilon$, $\epsilon \sim \mathcal{N}(0, \sigma)$, $p_m = 1/n$ |
| 6. Elitism    | Keep 1 best individual |
| 7. Repeat     | Until max generations |

In [ ]:
def genetic_algorithm(objective, bounds, pop_size=30, max_gen=50,
                       crossover_rate=0.9, mutation_rate=None,
                       mutation_sigma=0.1, tournament_k=2,
                       elite_count=1, seed=42):
    """
    Real-valued Genetic Algorithm with:
      - Tournament selection (k=2)
      - Arithmetic crossover
      - Additive Gaussian mutation
      - Elitism (preserve best individual)
    
    Returns: (best_params, best_cost, history)
    """
    rng = np.random.default_rng(seed)
    lower = np.array([b[0] for b in bounds])
    upper = np.array([b[1] for b in bounds])
    dim   = len(bounds)
    if mutation_rate is None:
        mutation_rate = 1.0 / dim
    sigma = mutation_sigma * (upper - lower)   # mutation strength per gene

    # --- 1. Initialize population uniformly in bounds ---
    pop = lower + rng.random((pop_size, dim)) * (upper - lower)
    fit = np.array([objective(ind) for ind in pop])

    best_idx = int(np.argmin(fit))
    best_x, best_f = pop[best_idx].copy(), float(fit[best_idx])
    history = []

    for gen in range(max_gen):
        # --- Elitism: preserve best individual ---
        elite_idx  = np.argsort(fit)[:elite_count]
        new_pop    = [pop[i].copy() for i in elite_idx]
        new_fit    = [fit[i] for i in elite_idx]

        while len(new_pop) < pop_size:
            # --- 3. Tournament selection ---
            def tournament():
                candidates = rng.choice(pop_size, tournament_k, replace=False)
                return pop[candidates[np.argmin(fit[candidates])]].copy()

            p1 = tournament()
            p2 = tournament()

            # --- 4. Arithmetic crossover ---
            if rng.random() < crossover_rate:
                alpha = rng.random()   # uniform blend weight
                child = alpha * p1 + (1 - alpha) * p2
            else:
                child = p1.copy()

            # --- 5. Additive Gaussian mutation (per-gene) ---
            mask = rng.random(dim) < mutation_rate
            child[mask] += rng.standard_normal(np.sum(mask)) * sigma[mask]

            # --- Clip to bounds (clipping method) ---
            child = np.clip(child, lower, upper)

            new_pop.append(child)
            new_fit.append(objective(child))

        pop = np.array(new_pop[:pop_size])
        fit = np.array(new_fit[:pop_size])

        gen_best = int(np.argmin(fit))
        if fit[gen_best] < best_f:
            best_f = float(fit[gen_best])
            best_x = pop[gen_best].copy()
        history.append(best_f)

    return best_x, best_f, history

print("Genetic Algorithm ready ✓")

### 6.3 Differential Evolution (DE)

Following the **course lecture on DE** — strategy **DE/rand/1/bin**:

| Step | Formula |
|------|---------|
| Initialization | $x_{j,i,1} = x_j^L + \text{rand} \cdot (x_j^U - x_j^L)$ |
| Mutation | $v_{i,G+1} = x_{r_1,G} + F \cdot (x_{r_2,G} - x_{r_3,G})$ |
| Crossover | $u_{j,i} = v_{j,i}$ if $\text{rand} \le CR$ or $j = I_{\text{rand}}$, else $x_{j,i}$ |
| Selection | $x_{i,G+1} = u_{i,G+1}$ if $f(u_{i,G+1}) \le f(x_{i,G})$, else $x_{i,G}$ |

In [ ]:
def differential_evolution(objective, bounds, pop_size=30, max_iter=50,
                            F=0.6, CR=0.9, seed=42):
    """
    Differential Evolution — DE/rand/1/bin strategy.

    Mutation:  v = x_r1 + F*(x_r2 - x_r3)
    Crossover: binomial (each gene from mutant with prob CR, one guaranteed)
    Selection: greedy (keep trial if better than target)
    Boundary:  reflection back inside bounds

    Returns: (best_params, best_cost, history)
    """
    rng   = np.random.default_rng(seed)
    lower = np.array([b[0] for b in bounds])
    upper = np.array([b[1] for b in bounds])
    dim   = len(bounds)
    NP    = max(pop_size, 4)   # DE requires NP >= 4

    def reflect(x):
        """Reflect out-of-bounds values back inside."""
        x = np.where(x < lower, 2*lower - x, x)
        x = np.where(x > upper, 2*upper - x, x)
        # Fallback: clip anything still outside
        return np.clip(x, lower, upper)

    # --- Initialization: uniform random in [lower, upper] ---
    pop = lower + rng.random((NP, dim)) * (upper - lower)
    fit = np.array([objective(p) for p in pop])

    best_idx = int(np.argmin(fit))
    best_x, best_f = pop[best_idx].copy(), float(fit[best_idx])
    history = []

    for it in range(max_iter):
        for i in range(NP):
            # Select 3 distinct random indices ≠ i
            others = [j for j in range(NP) if j != i]
            r1, r2, r3 = rng.choice(others, 3, replace=False)

            # --- Mutation: DE/rand/1 ---
            mutant = pop[r1] + F * (pop[r2] - pop[r3])
            mutant = reflect(mutant)

            # --- Binomial crossover ---
            # Ensure at least one gene comes from mutant (Irand trick)
            cross_mask = rng.random(dim) <= CR
            cross_mask[rng.integers(dim)] = True  # guarantee one gene
            trial = np.where(cross_mask, mutant, pop[i])

            # --- Greedy selection ---
            trial_f = objective(trial)
            if trial_f <= fit[i]:
                pop[i] = trial
                fit[i] = trial_f

        cur_best = int(np.argmin(fit))
        if fit[cur_best] < best_f:
            best_f = float(fit[cur_best])
            best_x = pop[cur_best].copy()
        history.append(best_f)

    return best_x, best_f, history

print("Differential Evolution ready ✓")

### 6.4 Particle Swarm Optimization (PSO)

Velocity update equation:
$$v_i[k+1] = w \cdot v_i[k] + c_1 r_1 (p_{\text{best},i} - x_i[k]) + c_2 r_2 (g_{\text{best}} - x_i[k])$$
$$x_i[k+1] = x_i[k] + v_i[k+1]$$

Where:
- $w$ = inertia weight (decreases from 0.9 → 0.4 linearly)
- $c_1$ = cognitive coefficient (pull toward personal best)
- $c_2$ = social coefficient (pull toward global best)
- $r_1, r_2 \sim U[0,1]$ random numbers

In [ ]:
def particle_swarm_optimization(objective, bounds, n_particles=30, max_iter=50,
                                 w_max=0.9, w_min=0.4, c1=1.5, c2=1.5,
                                 v_max_ratio=0.2, seed=42):
    """
    Particle Swarm Optimization (PSO).

    Velocity update:
        v = w*v + c1*r1*(pbest - x) + c2*r2*(gbest - x)
    Inertia w decreases linearly from w_max (0.9) to w_min (0.4).
    Velocities clamped to v_max = v_max_ratio * (upper - lower).

    Returns: (best_params, best_cost, history)
    """
    rng   = np.random.default_rng(seed)
    lower = np.array([b[0] for b in bounds])
    upper = np.array([b[1] for b in bounds])
    span  = upper - lower
    dim   = len(bounds)
    v_max = v_max_ratio * span

    # --- Initialize positions and velocities ---
    pos = lower + rng.random((n_particles, dim)) * span
    vel = rng.uniform(-1, 1, (n_particles, dim)) * v_max

    # Evaluate initial fitness
    fit = np.array([objective(p) for p in pos])

    p_best_pos = pos.copy()
    p_best_fit = fit.copy()
    g_best_idx = int(np.argmin(p_best_fit))
    g_best_pos = p_best_pos[g_best_idx].copy()
    g_best_fit = float(p_best_fit[g_best_idx])

    history = []

    for it in range(max_iter):
        # Linearly decreasing inertia weight
        w = w_max - (w_max - w_min) * it / max(1, max_iter - 1)

        r1 = rng.random((n_particles, dim))
        r2 = rng.random((n_particles, dim))

        # --- Velocity update ---
        cognitive = c1 * r1 * (p_best_pos - pos)
        social    = c2 * r2 * (g_best_pos  - pos)
        vel = w * vel + cognitive + social
        vel = np.clip(vel, -v_max, v_max)

        # --- Position update & clamp to bounds ---
        pos = np.clip(pos + vel, lower, upper)

        # --- Evaluate & update personal/global bests ---
        fit = np.array([objective(p) for p in pos])
        improved = fit < p_best_fit
        p_best_pos[improved] = pos[improved]
        p_best_fit[improved] = fit[improved]

        cur_best = int(np.argmin(p_best_fit))
        if p_best_fit[cur_best] < g_best_fit:
            g_best_fit = float(p_best_fit[cur_best])
            g_best_pos = p_best_pos[cur_best].copy()
        history.append(g_best_fit)

    return g_best_pos, g_best_fit, history

print("Particle Swarm Optimization ready ✓")

### 6.5 Simulated Annealing (SA)

Metropolis acceptance criterion:
$$P(\text{accept}) = \begin{cases} 1 & \text{if } \Delta f \le 0 \\ e^{-\Delta f / T} & \text{if } \Delta f > 0 \end{cases}$$

Temperature cooling schedule:
$$T_{k+1} = \alpha \cdot T_k \quad (\alpha < 1)$$

Proposal: Gaussian perturbation $x' = x + \mathcal{N}(0, \sigma) \cdot (x^U - x^L)$

In [ ]:
def simulated_annealing(objective, bounds, max_iter=300, T_init=1.0,
                         T_final=1e-3, cooling=0.98, step_size=0.1, seed=42):
    """
    Simulated Annealing — single-solution stochastic optimizer.

    Proposal: x' = x + N(0, step_size) * span  (reflected into bounds)
    Acceptance: always if f(x') < f(x), else with prob exp(-Δf / T)
    Cooling: geometric T_{k+1} = cooling * T_k

    Returns: (best_params, best_cost, history)
    """
    rng   = np.random.default_rng(seed)
    lower = np.array([b[0] for b in bounds])
    upper = np.array([b[1] for b in bounds])
    span  = upper - lower
    dim   = len(bounds)

    def reflect_into_bounds(x):
        """Reflect across boundaries to stay feasible."""
        result = x.copy()
        for _ in range(3):  # a few reflections handle large violations
            result = np.where(result < lower, 2*lower - result, result)
            result = np.where(result > upper, 2*upper - result, result)
        return np.clip(result, lower, upper)

    # --- Initialize ---
    current   = lower + rng.random(dim) * span
    cur_fit   = objective(current.tolist())
    best      = current.copy()
    best_fit  = cur_fit
    T         = T_init
    history   = []

    for it in range(max_iter):
        # --- Gaussian proposal step ---
        proposal = current + rng.standard_normal(dim) * step_size * span
        proposal = reflect_into_bounds(proposal)
        prop_fit = objective(proposal.tolist())

        # --- Metropolis acceptance ---
        delta = prop_fit - cur_fit
        if delta <= 0.0:
            current, cur_fit = proposal, prop_fit
        elif math.isfinite(delta):
            exponent = -delta / max(T, 1e-300)
            if exponent > -745 and rng.random() < math.exp(exponent):
                current, cur_fit = proposal, prop_fit

        # Update best
        if cur_fit < best_fit:
            best, best_fit = current.copy(), cur_fit

        history.append(float(best_fit))

        # --- Geometric cooling ---
        T *= cooling
        if T < T_final:
            break

    return best, best_fit, history

print("Simulated Annealing ready ✓")

### 6.6 Genetic Programming (GP) Optimizer

GP evolves **expression trees** (programs) rather than numeric parameter vectors.

| Step | Description |
|------|-------------|
| 1. Init | Ramped half-and-half: generate trees of varying depth |
| 2. Evaluate | Encode tree → numeric vector → compute objective |
| 3. Select | Tournament selection on fitness |
| 4. Crossover | Subtree crossover (swap random subtrees between parents) |
| 5. Mutate | Subtree replacement with a random new subtree |
| 6. Elitism | Preserve 1 best tree |

In [ ]:
# --- GP tree generation helpers ---

# --- Encode a GP tree back into the numeric heap-layout vector ---
# Inverse of _decode_gp_vector (defined with the GP Controller).
def encode_tree(tree, max_depth=GP_MAX_DEPTH):
    n_slots = 2**(max_depth + 1) - 1
    vec = np.zeros(2 * n_slots)
    all_sym  = list(FUNCS) + list(VARS) + ['const']
    leaf_sym = list(VARS)  + ['const']
    lo, hi = GP_CONST_RANGE

    def place(node, slot):
        is_leaf = _slot_depth(slot) == max_depth
        syms = leaf_sym if is_leaf else all_sym
        if isinstance(node, tuple):
            if is_leaf:
                raise ValueError("tree deeper than GP_MAX_DEPTH")
            sym, const, children = node[0], 0.0, node[1:]
        elif isinstance(node, str):
            sym, const, children = node, 0.0, ()
        else:
            sym, const, children = 'const', float(np.clip(node, lo, hi)), ()
        vec[2 * slot]     = syms.index(sym) + 0.5
        vec[2 * slot + 1] = const
        for k, ch in enumerate(children):
            place(ch, 2 * slot + 1 + k)

    place(tree, 0)
    return vec


def _random_terminal(rng):
    """Return a random variable or constant."""
    if rng.random() < 0.5:
        return rng.choice(list(VARS))
    lo, hi = GP_CONST_RANGE
    return float(rng.uniform(lo, hi))

def _grow_tree(rng, max_depth, current_depth=0):
    """Grow method: randomly create terminal or function node."""
    if current_depth >= max_depth or (current_depth > 0 and rng.random() < 0.4):
        return _random_terminal(rng)
    func = rng.choice(list(FUNCS))
    children = tuple(_grow_tree(rng, max_depth, current_depth + 1)
                     for _ in range(ARITY[func]))
    return (func,) + children

def _full_tree(rng, max_depth, current_depth=0):
    """Full method: always create function nodes until max_depth."""
    if current_depth >= max_depth:
        return _random_terminal(rng)
    func = rng.choice(list(FUNCS))
    children = tuple(_full_tree(rng, max_depth, current_depth + 1)
                     for _ in range(ARITY[func]))
    return (func,) + children

def _get_subtrees(tree):
    """Collect all subtrees and their parent info."""
    subtrees = [tree]
    if isinstance(tree, tuple):
        for child in tree[1:]:
            subtrees.extend(_get_subtrees(child))
    return subtrees

def _tree_depth(tree):
    if not isinstance(tree, tuple):
        return 0
    return 1 + max(_tree_depth(c) for c in tree[1:])

def _subtree_crossover(rng, parent1, parent2, max_depth=GP_MAX_DEPTH):
    """Create a child by inserting a random subtree from parent2 into parent1."""
    subs1 = _get_subtrees(parent1)
    subs2 = _get_subtrees(parent2)
    # Pick a random non-root insertion point in parent1
    idx = rng.integers(len(subs1))
    donor = subs2[rng.integers(len(subs2))]

    def replace(tree, target_idx, counter, replacement):
        if counter[0] == target_idx:
            counter[0] += 1
            return replacement
        counter[0] += 1
        if isinstance(tree, tuple):
            new_children = []
            for child in tree[1:]:
                new_children.append(replace(child, target_idx, counter, replacement))
            return (tree[0],) + tuple(new_children)
        return tree

    child = replace(parent1, idx, [0], donor)
    # Enforce depth limit
    if _tree_depth(child) > max_depth:
        return parent1  # reject and keep parent1
    return child

def _point_mutation(rng, tree):
    """Mutate a random node's function/terminal."""
    subs = _get_subtrees(tree)
    if len(subs) <= 1:
        return _random_terminal(rng)

    def mutate(node):
        if isinstance(node, tuple):
            name = rng.choice(list(FUNCS)) if rng.random() < 0.5 else node[0]
            # Adjust arity if needed
            n_children = ARITY[name]
            existing = list(node[1:])
            while len(existing) < n_children:
                existing.append(_random_terminal(rng))
            return (name,) + tuple(existing[:n_children])
        if isinstance(node, str):
            return rng.choice(list(VARS))
        # constant
        lo, hi = GP_CONST_RANGE
        return float(node) + rng.standard_normal() * 0.5

    # Pick a random node to mutate
    idx_to_mutate = rng.integers(len(subs))
    counter = [0]

    def traverse_mutate(tree):
        if counter[0] == idx_to_mutate:
            counter[0] += 1
            return mutate(tree)
        counter[0] += 1
        if isinstance(tree, tuple):
            return (tree[0],) + tuple(traverse_mutate(c) for c in tree[1:])
        return tree

    return traverse_mutate(tree)


def genetic_programming(objective, bounds, pop_size=20, max_gen=30,
                         crossover_rate=0.8, mutation_rate=0.3,
                         tournament_k=5, elite_count=1, seed=42):
    """
    Genetic Programming optimizer that evolves GP expression trees.
    Uses the numeric GP vector encoding to call the shared objective function.

    Returns: (best_vector, best_cost, history)
    """
    rng_std = random.Random(seed)
    rng_np  = np.random.default_rng(seed)

    # --- Ramped half-and-half initialization ---
    population = []
    for i in range(pop_size):
        depth = 1 + (i % GP_MAX_DEPTH)
        if i % 2 == 0:
            tree = _grow_tree(rng_np, depth)
        else:
            tree = _full_tree(rng_np, depth)
        population.append(tree)

    def eval_tree_obj(tree):
        try:
            vec = encode_tree(tree)
            return objective(vec)
        except Exception:
            return PENALTY

    fitness = [eval_tree_obj(t) for t in population]

    best_idx = int(np.argmin(fitness))
    best_tree = population[best_idx]
    best_f    = float(fitness[best_idx])
    history   = []

    for gen in range(max_gen):
        # Elitism
        order    = np.argsort(fitness)
        new_pop  = [population[i] for i in order[:elite_count]]
        new_fit  = [fitness[i] for i in order[:elite_count]]

        while len(new_pop) < pop_size:
            # Tournament selection
            def tournament():
                idx_list = rng_np.choice(pop_size, min(tournament_k, pop_size), replace=False).tolist()
                best_i = min(idx_list, key=lambda i: fitness[i])
                return population[best_i]

            p1 = tournament()
            p2 = tournament()

            # Crossover
            if rng_np.random() < crossover_rate:
                child = _subtree_crossover(rng_np, p1, p2)
            else:
                child = p1

            # Mutation
            if rng_np.random() < mutation_rate:
                child = _point_mutation(rng_np, child)

            child_f = eval_tree_obj(child)
            new_pop.append(child)
            new_fit.append(child_f)

        population = new_pop[:pop_size]
        fitness    = new_fit[:pop_size]

        cur_best = int(np.argmin(fitness))
        if fitness[cur_best] < best_f:
            best_f    = float(fitness[cur_best])
            best_tree = population[cur_best]
        history.append(best_f)

    # Return encoded vector of best tree
    best_vec = encode_tree(best_tree)
    return best_vec, best_f, history, best_tree


print("Genetic Programming Optimizer ready ✓")

---
## 7. Experiment Runner

The experiment matrix: **4 optimizers × 4 controllers × 3 plants = 48 runs**  
(+ 3 GP optimizer × GP controller × 3 plants runs when enabled)

In [ ]:
import copy

OPTIMIZERS = {
    'GA':  genetic_algorithm,
    'DE':  differential_evolution,
    'PSO': particle_swarm_optimization,
    'SA':  simulated_annealing,
}

CONTROLLERS = {
    'PID':            PIDController,
    'Neural Network': NeuralNetworkController,
    'Fuzzy':          FuzzyController,
    'GP Controller':  GPController,
}

# Budget per run — increase for better results (at the cost of time)
OPTIMIZER_KWARGS = {
    'GA':  dict(pop_size=20, max_gen=30, seed=42),
    'DE':  dict(pop_size=20, max_iter=30, F=0.6, CR=0.9, seed=42),
    'PSO': dict(n_particles=20, max_iter=30, seed=42),
    'SA':  dict(max_iter=200, T_init=1.0, T_final=1e-3, cooling=0.97, step_size=0.1, seed=42),
}

def run_single(opt_name, ctrl_name, plant_name):
    """Run one optimizer/controller/plant combination."""
    plant      = PLANTS[plant_name]
    controller = CONTROLLERS[ctrl_name]()
    optimizer  = OPTIMIZERS[opt_name]
    kwargs     = OPTIMIZER_KWARGS[opt_name].copy()
    bounds     = controller.parameter_bounds()
    obj_fn     = make_objective(controller, plant)

    t0 = time.perf_counter()
    best_x, best_f, history = optimizer(obj_fn, bounds, **kwargs)
    elapsed = time.perf_counter() - t0

    # Re-simulate with best parameters
    controller.set_parameters(best_x)
    t, y, u = simulate(plant, controller)
    metrics = compute_metrics(t, y)

    return {
        'optimizer':    opt_name,
        'controller':   ctrl_name,
        'plant':        plant_name,
        'best_params':  best_x,
        'best_fitness': best_f,
        'metrics':      metrics,
        'history':      history,
        't': t, 'y': y, 'u': u,
        'time_s':       elapsed,
        'settled':      metrics['settling_time'] is not None,
        'stable':       metrics['stable'],
    }

print("Experiment runner ready ✓")
print(f"Matrix: {len(OPTIMIZERS)} optimizers × {len(CONTROLLERS)} controllers × {len(PLANTS)} plants = "
      f"{len(OPTIMIZERS)*len(CONTROLLERS)*len(PLANTS)} runs")

---
## 8. Run Experiments

In [ ]:
results = []
total_start = time.perf_counter()
run_count = 0

for opt_name in OPTIMIZERS:
    for ctrl_name in CONTROLLERS:
        for plant_name in PLANTS:
            run_count += 1
            label = f"{opt_name}/{ctrl_name}/{plant_name}"
            try:
                record = run_single(opt_name, ctrl_name, plant_name)
                record['status'] = 'success'
                settled = '✓' if record['settled'] else '✗'
                m = record['metrics']
                ts_str = f"{m['settling_time']:.2f}s" if m['settling_time'] else 'N/A'
                print(f"[{run_count:2d}] {label:<50} | Settled:{settled} | IAE={m['iae']:.3f} | Ts={ts_str} | t={record['time_s']:.1f}s")
            except Exception as exc:
                record = {'optimizer': opt_name, 'controller': ctrl_name,
                          'plant': plant_name, 'status': 'failed',
                          'error': str(exc), 'metrics': {}, 'history': []}
                print(f"[{run_count:2d}] {label:<50} | FAILED: {exc}")
            results.append(record)

total_elapsed = time.perf_counter() - total_start
success_count = sum(1 for r in results if r.get('status') == 'success')
print(f"\n{'='*80}")
print(f"Completed: {success_count}/{len(results)} runs in {total_elapsed:.1f}s")
if total_elapsed > 300:
    print("⚠ WARNING: Total execution exceeded 5-minute target!")
else:
    print(f"✓ Within 5-minute limit ({total_elapsed:.1f}s / 300s)")

---
## 9. Results: Comparison Tables

In [ ]:
def print_comparison_table(results, plant_name):
    """Print a formatted comparison table for a given plant."""
    plant_results = [r for r in results
                     if r['plant'] == plant_name and r.get('status') == 'success']
    if not plant_results:
        print(f"No successful results for {plant_name}")
        return

    hdr  = f"{'Optimizer':<8} {'Controller':<18} {'Tr(s)':<8} {'Ts(s)':<8} {'OS%':<8} {'Ess':<8} {'IAE':<10} {'Fit':<10}"
    sep  = '-' * len(hdr)
    print(f"\n{'='*len(hdr)}")
    print(f" Plant: {plant_name.upper().replace('_', ' ')}")
    print(f"{'='*len(hdr)}")
    print(hdr)
    print(sep)

    for r in sorted(plant_results, key=lambda x: x.get('best_fitness', 1e9)):
        m  = r['metrics']
        tr = f"{m['rise_time']:.3f}" if m.get('rise_time') else 'N/A'
        ts = f"{m['settling_time']:.3f}" if m.get('settling_time') else 'N/A'
        os = f"{m.get('overshoot', 0):.2f}"
        ess= f"{m.get('sse', 0):.4f}"
        iae= f"{m.get('iae', 0):.4f}"
        fit= f"{r['best_fitness']:.4f}"
        print(f"{r['optimizer']:<8} {r['controller']:<18} {tr:<8} {ts:<8} {os:<8} {ess:<8} {iae:<10} {fit:<10}")
    print(sep)

for plant_name in PLANTS:
    print_comparison_table(results, plant_name)

---
## 10. Plots: Step Responses

In [ ]:
def plot_step_responses(results, plant_name, figsize=(14, 10)):
    """Plot step responses for all controllers and optimizers on one plant."""
    plant_results = [r for r in results
                     if r['plant'] == plant_name
                     and r.get('status') == 'success'
                     and 'y' in r
                     and np.any(np.isfinite(r['y']))]
    if not plant_results:
        print(f"No plottable results for {plant_name}")
        return

    ctrl_names = list(dict.fromkeys(r['controller'] for r in plant_results))
    n_ctrl = len(ctrl_names)
    n_opt  = len(OPTIMIZERS)
    colors = plt.cm.Set1(np.linspace(0, 0.8, n_opt))

    fig, axes = plt.subplots(1, n_ctrl, figsize=figsize, sharey=False)
    if n_ctrl == 1: axes = [axes]
    fig.suptitle(f'Step Responses — Plant: {plant_name.replace("_", " ").title()}',
                 fontsize=14, fontweight='bold')

    opt_list = list(OPTIMIZERS.keys())
    for ax, ctrl in zip(axes, ctrl_names):
        ctrl_results = [r for r in plant_results if r['controller'] == ctrl]
        for color, opt in zip(colors, opt_list):
            opt_res = [r for r in ctrl_results if r['optimizer'] == opt]
            if opt_res:
                r = opt_res[0]
                ax.plot(r['t'], r['y'], color=color, linewidth=1.5, label=opt)

        # Reference line
        t_max = max(r['t'][-1] for r in ctrl_results if 't' in r)
        ax.axhline(REFERENCE, color='black', linestyle='--', linewidth=1, label='Reference')
        ax.axhspan(REFERENCE*(1-SETTLING_TOL), REFERENCE*(1+SETTLING_TOL),
                   alpha=0.15, color='green', label='±2% band')
        ax.set_title(ctrl, fontsize=11)
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Output y(t)')
        ax.legend(fontsize=8, loc='lower right')
        ax.grid(True, alpha=0.3)
        ax.set_xlim(0, t_max)

    plt.tight_layout()
    plt.savefig(f'step_responses_{plant_name}.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Saved: step_responses_{plant_name}.png")

for plant_name in PLANTS:
    plot_step_responses(results, plant_name)

---
## 11. Plots: Optimizer Convergence Curves

In [ ]:
def plot_convergence(results, plant_name, ctrl_name, figsize=(10, 5)):
    """Plot convergence curves for all optimizers on one plant/controller combo."""
    subset = [r for r in results
              if r['plant'] == plant_name
              and r['controller'] == ctrl_name
              and r.get('status') == 'success'
              and len(r.get('history', [])) > 0]
    if not subset:
        return

    fig, ax = plt.subplots(figsize=figsize)
    for r in subset:
        h = np.array(r['history'])
        ax.plot(np.arange(1, len(h)+1), h, linewidth=1.8, label=r['optimizer'])

    ax.set_title(f'Convergence — {ctrl_name} on {plant_name.replace("_"," ").title()}',
                 fontsize=12)
    ax.set_xlabel('Iteration / Generation')
    ax.set_ylabel('Best Fitness (lower is better)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'convergence_{plant_name}_{ctrl_name.replace(" ","_")}.png',
                dpi=120, bbox_inches='tight')
    plt.show()

# Plot convergence for PID on each plant (most interpretable)
for plant_name in PLANTS:
    plot_convergence(results, plant_name, 'PID')
    plot_convergence(results, plant_name, 'Fuzzy')

---
## 12. GP Optimizer + GP Controller (Optional)

The GP optimizer evolves expression trees for the GP controller.

In [ ]:
# --- Run GP optimizer on GP controller (optional) ---
RUN_GP = True  # set to False to skip

if RUN_GP:
    gp_results = []
    for plant_name, plant in PLANTS.items():
        ctrl = GPController()
        bounds = ctrl.parameter_bounds()
        obj_fn = make_objective(ctrl, plant)

        print(f"Running GP optimizer on GP Controller / {plant_name}...")
        t0 = time.perf_counter()
        try:
            best_vec, best_f, history, best_tree = genetic_programming(
                obj_fn, bounds, pop_size=20, max_gen=20, seed=42
            )
            # Simulate best tree
            ctrl.set_parameters(best_vec)
            t, y, u = simulate(plant, ctrl)
            metrics = compute_metrics(t, y)
            elapsed = time.perf_counter() - t0
            print(f"  Best expression: {tree_to_str(best_tree)}")
            print(f"  Fitness: {best_f:.4f} | IAE: {metrics['iae']:.4f} | t={elapsed:.1f}s")

            gp_results.append({
                'optimizer': 'GP', 'controller': 'GP Controller',
                'plant': plant_name, 'status': 'success',
                'best_fitness': best_f, 'metrics': metrics,
                'history': history, 't': t, 'y': y, 'u': u,
                'expression': tree_to_str(best_tree), 'settled': metrics['settling_time'] is not None
            })
        except Exception as exc:
            print(f"  FAILED: {exc}")
            gp_results.append({'optimizer':'GP','controller':'GP Controller',
                                'plant':plant_name,'status':'failed','error':str(exc)})

    # Plot GP results
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
    fig.suptitle('GP Optimizer → GP Controller: Step Responses', fontsize=13, fontweight='bold')
    for ax, (pname, _) in zip(axes, PLANTS.items()):
        gp_r = [r for r in gp_results if r['plant'] == pname and r.get('status') == 'success']
        if gp_r:
            r = gp_r[0]
            ax.plot(r['t'], r['y'], 'b-', linewidth=2)
            ax.axhline(REFERENCE, color='k', linestyle='--', linewidth=1)
            ax.set_title(f"{pname.replace('_',' ').title()}\n{r.get('expression','')[:40]}", fontsize=9)
        ax.set_xlabel('Time (s)'); ax.set_ylabel('y(t)')
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('gp_step_responses.png', dpi=120, bbox_inches='tight')
    plt.show()

---
## 13. Summary: Best Combinations

In [ ]:
print("\n" + "="*80)
print(" SUMMARY: Best Optimizer-Controller Combination per Plant")
print("="*80)

for plant_name in PLANTS:
    plant_res = [
        r for r in results
        if r['plant'] == plant_name
        and r.get('status') == 'success'
        and r.get('metrics', {}).get('stable', False)
    ]
    if not plant_res:
        print(f"\n{plant_name}: No stable results.")
        continue

    # Sort by IAE (lower is better)
    plant_res.sort(key=lambda r: r.get('metrics', {}).get('iae', 1e9))
    best = plant_res[0]
    m = best['metrics']
    print(f"\n📌 {plant_name.upper().replace('_', ' ')}:")
    print(f"   Best: {best['optimizer']} + {best['controller']}")
    print(f"   IAE={m.get('iae',0):.4f} | OS={m.get('overshoot',0):.2f}% | "
          f"Ts={m.get('settling_time'):.3f}s" if m.get('settling_time') else
          f"   IAE={m.get('iae',0):.4f} | OS={m.get('overshoot',0):.2f}% | Ts=N/A")

print("\n" + "="*80)
print(f"Total execution time: {total_elapsed:.1f}s")
print("Outputs: step_responses_*.png, convergence_*.png")
print("="*80)

---
## 14. Export Results to CSV

In [ ]:
import csv, json

csv_path = 'experiment_results.csv'
fieldnames = ['optimizer','controller','plant','status','stable','settled',
              'rise_time','settling_time','overshoot','sse','iae',
              'best_fitness','time_s']

with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for r in results:
        m = r.get('metrics', {})
        writer.writerow({
            'optimizer':    r.get('optimizer',''),
            'controller':   r.get('controller',''),
            'plant':        r.get('plant',''),
            'status':       r.get('status',''),
            'stable':       m.get('stable',''),
            'settled':      r.get('settled',''),
            'rise_time':    m.get('rise_time',''),
            'settling_time':m.get('settling_time',''),
            'overshoot':    m.get('overshoot',''),
            'sse':          m.get('sse',''),
            'iae':          m.get('iae',''),
            'best_fitness': r.get('best_fitness',''),
            'time_s':       r.get('time_s',''),
        })

print(f"Results saved to: {csv_path}")
print(f"Total rows: {len(results)}")